# Run Stremio Telegram Addon on Google Colab

Exposes the local addon server to Stremio using an Ngrok tunnel.

Requirements:
- API ID & API Hash from my.telegram.org
- Ngrok authtoken from ngrok.com

In [ ]:
#@title Setup
!git clone -b beta https://github.com/SunilRoy-dev/stremio-telegram-debrid.git /content/app
%cd /content/app
!pip install -r requirements.txt tgcrypto pyngrok

In [ ]:
#@title Inputs
API_ID = "" #@param {type:"string"}
API_HASH = "" #@param {type:"string"}
TELEGRAM_CHANNEL_ID = "" #@param {type:"string"}
USER_SESSION_STRING = "" #@param {type:"string"}
BOT_TOKEN = "" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}
NGROK_AUTHTOKEN = "" #@param {type:"string"}

import os
os.environ["API_ID"] = API_ID.strip()
os.environ["API_HASH"] = API_HASH.strip()
os.environ["TELEGRAM_CHANNEL_ID"] = TELEGRAM_CHANNEL_ID.strip()
os.environ["USER_SESSION_STRING"] = USER_SESSION_STRING.strip()
os.environ["BOT_TOKEN"] = BOT_TOKEN.strip()
os.environ["API_KEY"] = API_KEY.strip()

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

In [ ]:
#@title Start Server
from pyngrok import ngrok
import subprocess
import time

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except:
    pass

public_tunnel = ngrok.connect(7860, "http")
addon_url = public_tunnel.public_url
os.environ["ADDON_URL"] = addon_url

manifest = "/manifest.json"
if API_KEY.strip():
    manifest = f"/manifest.json?api_key={API_KEY.strip()}"

print("Copy and paste this URL into Stremio:")
print(f"{addon_url}{manifest}")

p = subprocess.Popen(["uvicorn", "addon:app", "--host", "0.0.0.0", "--port", "7860"])
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    p.terminate()